# Identificación de especie con Pl@ntNet

Sugiere especie para las fotos de árboles/orquídeas/flores que quedaron en confianza "media" tras el escaneo del volumen (nombre común sin binomial, o carpetas administrativas como "Fichas"/"Articulos").

**No cubre aves ni insectos**, Pl@ntNet es solo plantas. Para esas ramas sigue el plan de iNaturalist (pendiente del profesor) o revisión manual.

**Insumo:** `../../experimentos/bd/campuslab_volumen.db` (no se mueve ni se duplica, se lee desde donde ya vive).
**Salida:** actualiza `especies.sugerido_por = 'PlantNet'` y dos columnas nuevas de auditoría (`plantnet_score`, `plantnet_raw`), nunca marca `fuente_validacion` (eso solo lo pone una confirmación humana/autorizada, no una sugerencia automática).

In [22]:
import json
import sqlite3
from pathlib import Path

import pandas as pd
import requests
from io import BytesIO
from PIL import Image

def cargar_config(archivo="config_local.json"):
    ruta = Path(archivo)
    if not ruta.exists():
        raise FileNotFoundError(
            f"No se encontro {archivo}. Copia config_local.example.json a {archivo} "
            "(junto a este notebook) y pon tu API key de Pl@ntNet. No se sube a git."
        )
    return json.loads(ruta.read_text(encoding="utf-8"))

config = cargar_config()
PLANTNET_API_KEY = config["plantnet_api_key"]
RUTA_BD = Path("../../experimentos/bd/campuslab_volumen.db")  # la BD real, sin mover ni duplicar

print(f"Base de datos: {RUTA_BD.resolve()} ({'existe' if RUTA_BD.exists() else 'NO ENCONTRADA'})")

Base de datos: C:\Users\hecto\OneDrive\Escritorio\ITESO\Comunidad\Campus Lab\experimentos\bd\campuslab_volumen.db (existe)


In [23]:
con = sqlite3.connect(RUTA_BD)

# Columnas de auditoria para no perder el detalle de que sugirio Pl@ntNet y con que confianza
cols = [r[1] for r in con.execute("PRAGMA table_info(especies)")]
for col, tipo in [("plantnet_score", "REAL"), ("plantnet_raw", "TEXT")]:
    if col not in cols:
        con.execute(f"ALTER TABLE especies ADD COLUMN {col} {tipo}")
con.commit()

# Candidatas: fotos ya migradas (tabla fotos, con ruta_actual real) que originalmente
# venian de una carpeta de especie en confianza media (staging_escaneo, via ruta_original).
# No se puede usar solo staging_escaneo para abrir el archivo: esas rutas ya no existen
# despues de que Migracion_Volumen.ipynb movio y renombro todo a Fotos_Unificadas.
consulta = """
    SELECT f.id AS foto_id, f.ruta_actual, s.carpeta_contenedora, s.especie_comun_sugerido
    FROM fotos f
    JOIN staging_escaneo s ON s.ruta_completa = f.ruta_original
    WHERE s.confianza_especie = ? AND f.especie_id IS NULL
"""
df_candidatas = pd.read_sql(consulta, con, params=("media",))

print(f"Total candidatas en confianza media: {len(df_candidatas):,}")

Total candidatas en confianza media: 2,179


## Función de identificación

Sube hasta 5 imágenes por solicitud (aquí mandamos de 1 en 1, más simple de auditar por foto). Según la documentación, la respuesta trae `results`, una lista ordenada por `score` con `species.scientificNameWithoutAuthor`, `species.family`, etc.

In [24]:
EXTENSIONES_SOPORTADAS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
LADO_MAXIMO_PX = 1024  # de sobra para identificacion, mantiene el archivo enviado chico

def preparar_imagen(ruta_imagen):
    """Abre, redimensiona si hace falta, y devuelve bytes JPEG listos para enviar.
    Devuelve None si el archivo no es un formato soportado (ej. RAW de camara, Thumbs.db)."""
    ext = Path(ruta_imagen).suffix.lower()
    if ext not in EXTENSIONES_SOPORTADAS:
        return None
    with Image.open(ruta_imagen) as img:
        img = img.convert("RGB")
        img.thumbnail((LADO_MAXIMO_PX, LADO_MAXIMO_PX))
        buffer = BytesIO()
        img.save(buffer, format="JPEG", quality=85)
        return buffer.getvalue()

def identificar_planta(ruta_imagen, organo="auto", api_key=PLANTNET_API_KEY):
    """Llama a Pl@ntNet con una sola imagen. Devuelve (nombre_cientifico, score, respuesta_completa).
    Si el archivo no es un formato soportado, devuelve (None, None, {"omitido": "formato no soportado"})
    sin gastar una llamada a la API."""
    imagen_bytes = preparar_imagen(ruta_imagen)
    if imagen_bytes is None:
        return None, None, {"omitido": f"formato no soportado: {Path(ruta_imagen).suffix}"}

    url = f"https://my-api.plantnet.org/v2/identify/all?api-key={api_key}"
    archivos = {"images": (Path(ruta_imagen).stem + ".jpg", imagen_bytes, "image/jpeg")}
    datos = {"organs": organo}
    resp = requests.post(url, files=archivos, data=datos, timeout=30)

    if not resp.ok:
        # Antes solo se veia el codigo (ej. "404 Client Error"). Esto guarda el mensaje real
        # del servidor, que normalmente explica la causa concreta.
        detalle = resp.text[:500]
        raise requests.HTTPError(f"{resp.status_code} {resp.reason}: {detalle}", response=resp)

    cuerpo = resp.json()
    if cuerpo.get("results"):
        mejor = cuerpo["results"][0]
        return mejor["species"]["scientificNameWithoutAuthor"], mejor["score"], cuerpo
    return None, None, cuerpo

## Prueba de parseo (sin llamar a la API real)

No tengo acceso de red a plantnet.org desde este entorno, así que antes de que la corras contra la API real, valido que el parseo de la respuesta funcione contra la forma exacta de JSON que documenta Pl@ntNet.

In [25]:
RESPUESTA_EJEMPLO_DOCUMENTACION = {
    "query": {"project": "all", "images": ["abc"], "organs": ["auto"]},
    "language": "en",
    "preferedReferential": "k-world-flora",
    "bestMatch": "Ajuga genevensis L.",
    "results": [
        {
            "score": 0.90734,
            "species": {
                "scientificNameWithoutAuthor": "Ajuga genevensis",
                "scientificNameAuthorship": "L.",
                "genus": {"scientificNameWithoutAuthor": "Ajuga"},
                "family": {"scientificNameWithoutAuthor": "Lamiaceae"},
            },
        }
    ],
}

def parsear_respuesta(cuerpo):
    if cuerpo.get("results"):
        mejor = cuerpo["results"][0]
        return mejor["species"]["scientificNameWithoutAuthor"], mejor["score"]
    return None, None

nombre, score = parsear_respuesta(RESPUESTA_EJEMPLO_DOCUMENTACION)
assert nombre == "Ajuga genevensis" and abs(score - 0.90734) < 1e-6
print(f"Parseo correcto: {nombre} (score {score})")

nombre_vacio, score_vacio = parsear_respuesta({"results": []})
assert nombre_vacio is None and score_vacio is None
print("Caso sin resultados manejado correctamente (no truena, regresa None)")

Parseo correcto: Ajuga genevensis (score 0.90734)
Caso sin resultados manejado correctamente (no truena, regresa None)


## Piloto real (20-30 fotos)

**No lo corras hasta tener tu API key en `config_local.json`.** Empieza con un lote chico para validar calidad de las sugerencias antes de lanzarlo contra las ~2,166 candidatas completas. Pl@ntNet tiene cuota diaria (se ve en `remainingIdentificationRequests` de cada respuesta), así que un lote grande de una sola vez podría no caber en un día.

In [26]:
TAMANO_PILOTO = 25
EJECUTAR_PILOTO_REAL = True  # <-- cambialo a True cuando tengas la API key

if not EJECUTAR_PILOTO_REAL:
    print("EJECUTAR_PILOTO_REAL esta en False. No se llamo a la API. Pon tu API key y activalo cuando estes lista.")
else:
    muestra = df_candidatas.sample(min(TAMANO_PILOTO, len(df_candidatas)), random_state=1)
    resultados = []
    for _, fila in muestra.iterrows():
        try:
            nombre, score, crudo = identificar_planta(fila["ruta_actual"])
            resultados.append({"id": fila["foto_id"], "sugerido": nombre, "score": score, "error": None})
            if nombre:
                # Busca o crea la especie sugerida, y enlaza esta foto especifica a ella
                especie_row = con.execute("SELECT id FROM especies WHERE nombre_cientifico=?", (nombre,)).fetchone()
                if especie_row:
                    especie_id = especie_row[0]
                    con.execute(
                        "UPDATE especies SET sugerido_por=?, plantnet_score=?, plantnet_raw=? WHERE id=?",
                        ("PlantNet", score, json.dumps(crudo, ensure_ascii=False), especie_id)
                    )
                else:
                    cur = con.execute(
                        "INSERT INTO especies (nombre_cientifico, sugerido_por, plantnet_score, plantnet_raw) VALUES (?,?,?,?)",
                        (nombre, "PlantNet", score, json.dumps(crudo, ensure_ascii=False))
                    )
                    especie_id = cur.lastrowid
                con.execute("UPDATE fotos SET especie_id=?, estado=? WHERE id=?", (especie_id, "pendiente", fila["foto_id"]))
        except Exception as e:
            resultados.append({"id": fila["foto_id"], "sugerido": None, "score": None, "error": str(e)})
    con.commit()

    df_resultado = pd.DataFrame(resultados)
    print(df_resultado.to_string(index=False))
    print(f"\nExitosos: {df_resultado['error'].isna().sum()} / {len(df_resultado)}")

# Nota: la conexion se deja abierta a proposito, para poder re-correr esta celda
# (reintentar el piloto, por ejemplo) sin tener que re-ejecutar todo el notebook desde el inicio.

  id                sugerido   score error
4044   Boscia mossambicensis 0.17197  None
2168       Stanhopea oculata 0.26263  None
2927      Terminalia catappa 0.97564  None
3679     Citrus × microcarpa 0.41892  None
3627                    None     NaN  None
1239      Citrus × aurantium 0.79922  None
1498           Plumeria alba 0.88752  None
1354    Araucaria columnaris 0.60329  None
1959    Pennellia longifolia 0.08003  None
1836   Spathoglottis plicata 0.48904  None
3624      Ficus maclellandii 0.30543  None
1395      Terminalia mantaly 0.13097  None
4166          Tabebuia rosea 0.44954  None
2044       Oncidium hastatum 0.20443  None
1496           Plumeria alba 0.97081  None
2960    Araucaria columnaris 0.29956  None
3753   Byrsonima crassifolia 0.13783  None
4094   Jacaranda mimosifolia 0.38203  None
3225 Pseudobombax ellipticum 0.75189  None
3396    Fouquieria splendens 0.13549  None
3370          Cojoba arborea 0.42915  None
2325        Stanhopea wardii 0.20365  None
2278 Stanho